# DSA210 — Notebook 1: Data Cleaning & Merging

**What this notebook does:**
1. Loads charging station counts from Donanım Haber (March 2026 xlsx)
2. Loads province population from TÜİK ADNKS (2025)
3. Loads province vehicle counts from TÜİK (2026)
4. Loads national EV time-series from TÜİK
5. Cleans province names, merges all sources, engineers features
6. Saves `master.csv` and `ev_timeseries.csv` for Notebooks 2 & 3

**Output files:** `master.csv`, `ev_timeseries.csv`

In [13]:
import warnings
warnings.filterwarnings('ignore')

import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path('.')
FIGURES = BASE / 'figures'
FIGURES.mkdir(exist_ok=True)

print('Libraries loaded OK')

Libraries loaded OK


---
## Helpers: Province Name Normalisation

Two functions work together:

- **`clean_name`** — fixes encoding artifacts (U+0307 combining dot, a TÜİK artefact) and title-cases the name for display. Turkish characters are preserved.
- **`key_name`** — converts the name to **lowercase ASCII** for merging. This avoids mismatches caused by different encodings of İ/I, ş/s, etc. across files. A short `ALIAS_MAP` resolves common abbreviations ("Afyon" → "afyonkarahisar", "K.Maraş" → "kahramanmaras").

In [14]:
# Turkish-specific characters → ASCII equivalents (for the merge key)
# Only the truly Turkish chars that Python's str.lower() doesn't handle correctly
TR_MAP = str.maketrans(
    'İŞĞÜÖÇışğüöç',
    'isguocisguoc'
)

# Known short-forms or misspellings that appear in source files
ALIAS_MAP = {
    'afyon':          'afyonkarahisar',
    'k.maras':        'kahramanmaras',   # handles K.Maraş after TR_MAP
    'kahraman maras': 'kahramanmaras',
    
}


def clean_name(name):
    """Fix encoding artifacts and title-case for display. Keeps Turkish chars."""
    if not isinstance(name, str):
        return str(name)
    
    

    name = unicodedata.normalize('NFC', name)
    name = name.replace('\u0307', '')   # TÜİK sometimes embeds combining dot above
    return name.strip().title()


def key_name(name):
    """Return a lowercase ASCII key used only for cross-dataset merging."""
    if not isinstance(name, str):
        name = str(name)
    name = unicodedata.normalize('NFC', name)
    name = name.replace('\u0307', '')
    name = name.strip()
    name = name.translate(TR_MAP)       # Turkish caps → ASCII before lower()
    name = name.lower()                # remaining ASCII chars
    name = name.replace(" ", "")      # since i encounter problem in data : Ka Rs -> should be kars
    return ALIAS_MAP.get(name, name)    # resolve abbreviations


# Sanity checks
print(clean_name('Kocaeli\u0307'))    # → Kocaeli   (encoding artifact removed)
print(clean_name('ISTANBUL'))         # → Istanbul  (display only, not the merge key)
print(key_name('İSTANBUL'))           # → istanbul
print(key_name('ISTANBUL'))           # → istanbul
print(key_name('K.Maraş'))            # → kahramanmaras  (alias resolved)
print(key_name('Afyon'))              # → afyonkarahisar (alias resolved)

Kocaeli
Istanbul
istanbul
istanbul
kahramanmaras
afyonkarahisar


### Numbers cleaning 
#### e.g. (fix "455 042" issue)

---
## Step 1 · Charging Stations (Donanım Haber, March 2026)

Source: `(Mart 2026) Hangi şehirde kaç tane elektrikli araç şarj istasyonu var_.xlsx`  
Columns: Province | Total_Stations | Change_Pct | Car_Count | EV_Count | Station_Ratio

In [15]:
raw_stations = pd.read_excel(
    BASE / '(Mart 2026) Hangi şehirde kaç tane elektrikli araç şarj istasyonu var_.xlsx',
    header=0,
    engine='openpyxl'
)

# Print raw column names BEFORE renaming so we can confirm which column is EV_Count
# This guards against accidentally picking up the station count as EV_Count
print('Raw column names:', raw_stations.columns.tolist())
print('\nFirst 3 rows (raw):')
print(raw_stations.head(3).to_string())

# Rename to standard English names
raw_stations.columns = [
    'Province', 'Total_Stations', 'Change_Pct',
    'Car_Count', 'EV_Count', 'Station_Ratio'
]

df_stations = raw_stations[['Province', 'Total_Stations', 'EV_Count']].copy()
df_stations['Province']       = df_stations['Province'].astype(str).apply(clean_name)
df_stations['_key']           = df_stations['Province'].apply(key_name)   # for merging
df_stations['Total_Stations'] = (
    df_stations['Total_Stations']
    .astype(str)
    .str.replace(" ", "", regex=False)
)

df_stations['Total_Stations'] = pd.to_numeric(df_stations['Total_Stations'], errors='coerce')
df_stations['EV_Count'] = (
    df_stations['EV_Count']
    .astype(str)
    .str.replace(" ", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", "", regex=False)
)

df_stations['EV_Count'] = pd.to_numeric(df_stations['EV_Count'], errors='coerce')
print(f'\nStations data: {len(df_stations)} provinces')
print(f'Total stations across Turkey: {df_stations["Total_Stations"].sum():,}')
print(f'Total EVs across Turkey:      {df_stations["EV_Count"].sum():,}')

# Major cities must have non-zero EV counts — if any is 0 the source column is wrong
print('\n--- EV_Count source validation ---')
for city in ['istanbul', 'ankara', 'izmir']:
    row = df_stations[df_stations['_key'] == city]
    if row.empty:
        print(f'  NOT FOUND: {city}')
    elif row['EV_Count'].values[0] == 0:
        print(f'  WARNING: {city} has EV_Count = 0 — check which raw column is EV_Count!')
    else:
        ev = row['EV_Count'].values[0]
        st = row['Total_Stations'].values[0]
        print(f'  OK: {city}  EV_Count={ev:,}  Total_Stations={st:,}')

df_stations.head(8)

Raw column names: ['Şehir (İl)', 'Şarj İstasyonu Sayısı', 'İstasyon Sayısındaki Değişim Oranı (%)', 'Otomobil Sayısı', 'İldeki Elektrikli Araç Sayısı (Tahmini)', 'Şarj İstasyonu Oranı (Her 100 Araca)']

First 3 rows (raw):
       Şehir (İl)  Şarj İstasyonu Sayısı  İstasyon Sayısındaki Değişim Oranı (%) Otomobil Sayısı İldeki Elektrikli Araç Sayısı (Tahmini)  Şarj İstasyonu Oranı (Her 100 Araca)
0           Adana                    222                                  0.0183         455 042                                  10 011                                  2.22
1        Adıyaman                     29                                  0.0000          70 781                                   1 557                                  1.86
2  Afyonkarahisar                    159                                  0.0000         118 124                                   2 599                                  6.12

Stations data: 81 provinces
Total stations across Turkey: 14,811
Total EVs a

,Province,Total_Stations,EV_Count,_key
0,Adana,222,10011,adana
1,Adıyaman,29,1557,adiyaman
2,Afyonkarahisar,159,2599,afyonkarahisar
3,Ağrı,16,251,agri
4,Aksaray,55,1942,aksaray
5,Amasya,46,1684,amasya
6,Ankara,1777,47981,ankara
7,Antalya,886,16715,antalya


---
## Step 2 · Province Population (TÜİK ADNKS, 2025)

In [16]:
raw_pop = pd.read_excel(
    BASE / 'Cinsiyete Göre Nüfus_nip.tuik.xlsx',
    header=1,
    engine='openpyxl'
)

raw_pop.columns = ['Year', 'Province', 'Population', 'Male', 'Female']

df_population = raw_pop[['Year', 'Province', 'Population']].copy()
df_population['Year']       = pd.to_numeric(df_population['Year'], errors='coerce')
df_population['Population'] = pd.to_numeric(df_population['Population'], errors='coerce')
df_population = df_population.dropna(subset=['Year', 'Population'])
df_population = df_population[df_population['Year'] == 2025]
df_population['Province'] = df_population['Province'].astype(str).apply(clean_name)
df_population['_key']     = df_population['Province'].apply(key_name)   # for merging
df_population = df_population[['Province', '_key', 'Population']].reset_index(drop=True)

print(f'Population data: {len(df_population)} provinces')

# Duplicates would silently inflate population after a merge
dups = df_population[df_population['_key'].duplicated(keep=False)]
if not dups.empty:
    print('WARNING: Duplicate province keys in population data:')
    print(dups[['Province', '_key']].to_string())
else:
    print('OK: No duplicate provinces in population data')

df_population.head(8)

Population data: 81 provinces
OK: No duplicate provinces in population data


,Province,_key,Population
0,Adana,adana,2283609.0
1,Adiyaman,adiyaman,617821.0
2,Afyonkarahi̇sar,afyonkarahisar,751808.0
3,Ağri,agri,491489.0
4,Aksaray,aksaray,441136.0
5,Amasya,amasya,342242.0
6,Ankara,ankara,5910320.0
7,Antalya,antalya,2777677.0


---
## Step 3 · Province Vehicle Counts (TÜİK, 2026)

This file uses a two-panel layout: left half (cols 0–10) and right half (cols 13–23) are stacked.

In [17]:
raw_veh = pd.read_excel(
    BASE / 'İllere göre motorlu kara taşıtları sayısı .xls',
    sheet_name=0, header=None, engine='xlrd'
)

col_names = ['Province', 'Total_Vehicles', 'Dist_Pct', 'Car',
             'Minibus', 'Bus', 'Small_Truck', 'Truck',
             'Motorcycle', 'Special', 'Tractor']

# The Excel has a two-panel layout; stack both halves into one table
left  = raw_veh.iloc[6:, 0:11].copy()
right = raw_veh.iloc[6:, 13:24].copy()
left.columns = right.columns = col_names

df_vehicles = pd.concat([left, right], ignore_index=True)
df_vehicles = df_vehicles[df_vehicles['Province'].notna()]
df_vehicles = df_vehicles[~df_vehicles['Province'].astype(str).str.startswith('Toplam')]
df_vehicles = df_vehicles[~df_vehicles['Province'].astype(str).str.contains(',')]
df_vehicles = df_vehicles[~df_vehicles['Province'].astype(str).str.match(r'^[\(\d]')]
df_vehicles = df_vehicles[pd.to_numeric(df_vehicles['Total_Vehicles'], errors='coerce').notna()]

df_vehicles['Province']       = df_vehicles['Province'].astype(str).apply(clean_name)
df_vehicles['_key']           = df_vehicles['Province'].apply(key_name)   # for merging
df_vehicles['Total_Vehicles'] = pd.to_numeric(df_vehicles['Total_Vehicles'], errors='coerce')
df_vehicles['Car']            = pd.to_numeric(df_vehicles['Car'], errors='coerce')
df_vehicles = df_vehicles[['Province', '_key', 'Total_Vehicles', 'Car']].reset_index(drop=True)

print(f'Vehicle data: {len(df_vehicles)} provinces')

dups = df_vehicles[df_vehicles['_key'].duplicated(keep=False)]
if not dups.empty:
    print('WARNING: Duplicate province keys in vehicle data:')
    print(dups[['Province', '_key']].to_string())
else:
    print('OK: No duplicate provinces in vehicle data')

df_vehicles.head(8)

Vehicle data: 81 provinces
OK: No duplicate provinces in vehicle data


,Province,_key,Total_Vehicles,Car
0,Adana,adana,961667,455042
1,Adıyaman,adiyaman,162989,70781
2,Afyonkarahisar,afyonkarahisar,325817,118124
3,Ağrı,agri,37105,11394
4,Amasya,amasya,164371,76550
5,Ankara,ankara,3030967,2180956
6,Antalya,antalya,1689117,759753
7,Artvin,artvin,50497,21586


---
## Step 4 · National EV Time-Series (TÜİK, 2004–2025)

Column layout: Year | Total | % | Gasoline | % | Diesel | % | LPG | % | Hybrid | % | **Electric** | % | Unknown | %

In [18]:
# EV share (%) = electric vehicles / total cars
# Shows how much of the total vehicle fleet is electric over time
# Useful to track EV adoption growth in Turkey




raw_ts = pd.read_excel(
    BASE / 'Trafiğe Kayıtlı Otomobillerin Yakıt Cinsine Göre Dağılımı .xls',
    sheet_name=0, header=None, engine='xlrd'
)

ts = raw_ts.iloc[4:].copy()
ts.columns = range(ts.shape[1])
ts[0]  = pd.to_numeric(ts[0],  errors='coerce')  # Year
ts[1]  = pd.to_numeric(ts[1],  errors='coerce')  # Total cars
ts[11] = pd.to_numeric(ts[11], errors='coerce')  # Electric
ts = ts.dropna(subset=[0, 1]).fillna({11: 0})
ts = ts.rename(columns={0: 'Year', 1: 'Total_Cars', 11: 'Electric'})
ts['Year'] = ts['Year'].astype(int)
ts['EV_Share_Pct'] = ts['Electric'] / ts['Total_Cars'] * 100

df_ev_ts = ts[['Year', 'Total_Cars', 'Electric', 'EV_Share_Pct']].reset_index(drop=True)
df_ev_ts.to_csv(BASE / 'ev_timeseries.csv', index=False)
print('Saved ev_timeseries.csv')
df_ev_ts.tail(8)

Saved ev_timeseries.csv


,Year,Total_Cars,Electric,EV_Share_Pct
14,2018,12398190.0,952.0,0.007679
15,2019,12503049.0,1176.0,0.009406
16,2020,13099041.0,2797.0,0.021353
17,2021,13706065.0,6267.0,0.045724
18,2022,14269352.0,14552.0,0.101981
19,2023,15221134.0,80043.0,0.525868
20,2024,16232458.0,183776.0,1.132151
21,2025,17373581.0,370591.0,2.133072


---
## Step 5 · Merge All Sources

In [19]:
# Merge on the ASCII _key so Turkish character differences never break the join
df_merged = df_stations.merge(df_population[['_key', 'Population']], on='_key', how='left')
df_merged = df_merged.merge(df_vehicles[['_key', 'Total_Vehicles', 'Car']],   on='_key', how='left')

# --- LEFT side: provinces in stations that got no match ---
missing_pop = df_merged[df_merged['Population'].isna()]['Province'].tolist()
missing_veh = df_merged[df_merged['Total_Vehicles'].isna()]['Province'].tolist()
if missing_pop:
    print(f'WARNING – Population missing for: {missing_pop}')
else:
    print('OK: All station provinces matched a population row')
if missing_veh:
    print(f'WARNING – Vehicles missing for: {missing_veh}')
else:
    print('OK: All station provinces matched a vehicles row')

# --- RIGHT side: provinces in population/vehicles that never matched a station ---
keys_stations = set(df_stations['_key'])
extra_pop = set(df_population['_key']) - keys_stations
extra_veh = set(df_vehicles['_key'])   - keys_stations
if extra_pop:
    print(f'\nProvinces in POPULATION but NOT in stations: {sorted(extra_pop)}')
if extra_veh:
    print(f'Provinces in VEHICLES but NOT in stations:   {sorted(extra_veh)}')

print(f'\nMerged shape: {df_merged.shape}')
print(f'Unique provinces: {df_merged["Province"].nunique()} (expected 81)')

# Drop the merge key — it was only needed for joining
df_merged = df_merged.drop(columns=['_key'])

df_merged.head(5)

OK: All station provinces matched a population row
OK: All station provinces matched a vehicles row

Merged shape: (81, 7)
Unique provinces: 81 (expected 81)


,Province,Total_Stations,EV_Count,Population,Total_Vehicles,Car
0,Adana,222,10011,2283609.0,961667,455042
1,Adıyaman,29,1557,617821.0,162989,70781
2,Afyonkarahisar,159,2599,751808.0,325817,118124
3,Ağrı,16,251,491489.0,37105,11394
4,Aksaray,55,1942,441136.0,183803,88275


---
## Step 6 · Add Region & Metropolitan Flag

In [ ]:
# Keys are lowercase ASCII (output of key_name) so the map is encoding-independent
REGION_MAP = {
    # Marmara
    'istanbul':'Marmara', 'tekirdag':'Marmara', 'edirne':'Marmara',
    'kirklareli':'Marmara', 'balikesir':'Marmara', 'canakkale':'Marmara',
    'bursa':'Marmara', 'kocaeli':'Marmara', 'sakarya':'Marmara',
    'duzce':'Marmara', 'bolu':'Marmara', 'yalova':'Marmara','bilecik': 'Marmara',
    # Aegean
    'izmir':'Aegean', 'manisa':'Aegean', 'aydin':'Aegean', 'denizli':'Aegean',
    'mugla':'Aegean', 'kutahya':'Aegean', 'afyonkarahisar':'Aegean', 'usak':'Aegean',
    # Mediterranean
    'antalya':'Mediterranean', 'isparta':'Mediterranean', 'burdur':'Mediterranean',
    'adana':'Mediterranean', 'mersin':'Mediterranean', 'hatay':'Mediterranean',
    'kahramanmaras':'Mediterranean', 'osmaniye':'Mediterranean',
    # Central Anatolia
    'ankara':'Central Anatolia', 'konya':'Central Anatolia', 'eskisehir':'Central Anatolia',
    'kirikkale':'Central Anatolia', 'kirsehir':'Central Anatolia', 'nevsehir':'Central Anatolia',
    'aksaray':'Central Anatolia', 'nigde':'Central Anatolia', 'karaman':'Central Anatolia',
    'sivas':'Central Anatolia', 'kayseri':'Central Anatolia', 'yozgat':'Central Anatolia',
    'cankiri':'Central Anatolia',
    # Black Sea
    'zonguldak':'Black Sea', 'karabuk':'Black Sea', 'bartin':'Black Sea',
    'kastamonu':'Black Sea', 'sinop':'Black Sea', 'samsun':'Black Sea',
    'ordu':'Black Sea', 'giresun':'Black Sea', 'trabzon':'Black Sea',
    'rize':'Black Sea', 'artvin':'Black Sea', 'amasya':'Black Sea',
    'tokat':'Black Sea', 'corum':'Black Sea', 'gumushane':'Black Sea',
    'bayburt':'Black Sea',
    # Eastern Anatolia
    'malatya':'Eastern Anatolia', 'elazig':'Eastern Anatolia', 'bingol':'Eastern Anatolia',
    'tunceli':'Eastern Anatolia', 'van':'Eastern Anatolia', 'mus':'Eastern Anatolia',
    'bitlis':'Eastern Anatolia', 'hakkari':'Eastern Anatolia', 'agri':'Eastern Anatolia',
    'igdir':'Eastern Anatolia', 'kars':'Eastern Anatolia', 'ardahan':'Eastern Anatolia',
    'erzincan':'Eastern Anatolia', 'erzurum':'Eastern Anatolia',
    # Southeastern Anatolia
    'gaziantep':'Southeastern Anatolia', 'adiyaman':'Southeastern Anatolia',
    'kilis':'Southeastern Anatolia', 'sanliurfa':'Southeastern Anatolia',
    'diyarbakir':'Southeastern Anatolia', 'mardin':'Southeastern Anatolia',
    'batman':'Southeastern Anatolia', 'sirnak':'Southeastern Anatolia',
    'siirt':'Southeastern Anatolia',
}

# Turkey's 30 büyükşehir belediyesi / also as lowercase ASCII keys
METROPOLITAN = {
    'adana', 'ankara', 'antalya', 'aydin', 'balikesir', 'bursa',
    'denizli', 'diyarbakir', 'erzurum', 'eskisehir', 'gaziantep',
    'hatay', 'istanbul', 'izmir', 'kahramanmaras', 'kayseri',
    'kocaeli', 'konya', 'malatya', 'manisa', 'mardin', 'mersin',
    'mugla', 'ordu', 'sakarya', 'samsun', 'sanliurfa', 'tekirdag',
    'trabzon', 'van',
}

# Apply mappings via key_name so the display name format doesn't matter
df_merged['Region']          = df_merged['Province'].apply(key_name).map(REGION_MAP)
df_merged['Is_Metropolitan'] = df_merged['Province'].apply(key_name).isin(METROPOLITAN).astype(int)

unmapped = df_merged[df_merged['Region'].isna()]['Province'].tolist()
if unmapped:
    print(f'WARNING – no region assigned to: {unmapped}')
else:
    print('OK: All provinces mapped to a region')

print(f"Metropolitan: {df_merged['Is_Metropolitan'].sum()}  |  Non-metro: {(df_merged['Is_Metropolitan']==0).sum()}")

OK: All provinces mapped to a region
Metropolitan: 30  |  Non-metro: 51


---
## Step 7 · Consistency Checks + Feature Engineering

In [21]:
# --- Consistency checks (run before computing any derived features) ---

# 1. Province count — Turkey has exactly 81 provinces
n_provinces = df_merged['Province'].nunique()
if n_provinces != 81:
    print(f'WARNING: Expected 81 provinces, got {n_provinces}')
else:
    print(f'OK: Province count = {n_provinces}')

# 2. No duplicate province rows
dups = df_merged[df_merged['Province'].duplicated(keep=False)]
if not dups.empty:
    print('WARNING: Duplicate provinces in master table:')
    print(dups['Province'].tolist())
else:
    print('OK: No duplicate provinces')

# 3. Missing values in key columns
for col in ['EV_Count', 'Total_Stations', 'Population', 'Total_Vehicles']:
    n_missing = df_merged[col].isna().sum()
    if n_missing > 0:
        prov = df_merged[df_merged[col].isna()]['Province'].tolist()
        print(f'WARNING: {col} has {n_missing} NaN(s) — {prov}')
    else:
        print(f'OK: {col} — complete')

# 4. Numeric columns should actually be numeric
for col in ['EV_Count', 'Total_Stations', 'Population', 'Total_Vehicles']:
    if not pd.api.types.is_numeric_dtype(df_merged[col]):
        print(f'WARNING: {col} is not numeric (dtype: {df_merged[col].dtype})')

print()

# --- Feature Engineering ---

# Stations per 100,000 residents — primary per-capita metric
df_merged['Stations_Per_Capita'] = (
    df_merged['Total_Stations'] / df_merged['Population'] * 100_000
)

# Per-10k normalised metrics for scatter plots
df_merged['EVs_Per_10k']      = df_merged['EV_Count']       / df_merged['Population'] * 10_000
df_merged['Stations_Per_10k'] = df_merged['Total_Stations'] / df_merged['Population'] * 10_000

df_merged = df_merged.sort_values('Province').reset_index(drop=True)

print(df_merged[['Stations_Per_Capita', 'EVs_Per_10k', 'Stations_Per_10k']].describe().round(3))
df_merged.head(5)

OK: Province count = 81
OK: No duplicate provinces
OK: EV_Count — complete
OK: Total_Stations — complete
OK: Population — complete
OK: Total_Vehicles — complete

       Stations_Per_Capita  EVs_Per_10k  Stations_Per_10k
count               81.000       81.000            81.000
mean                13.472       35.188             1.347
std                  7.783       16.139             0.778
min                  3.255        2.646             0.326
25%                  7.735       25.201             0.774
50%                 12.820       38.841             1.282
75%                 16.120       45.218             1.612
max                 39.562       81.182             3.956


,Province,Total_Stations,EV_Count,Population,Total_Vehicles,Car,Region,Is_Metropolitan,Stations_Per_Capita,EVs_Per_10k,Stations_Per_10k
0,Adana,222,10011,2283609.0,961667,455042,Mediterranean,1,9.721454,43.838503,0.972145
1,Adıyaman,29,1557,617821.0,162989,70781,Southeastern Anatolia,0,4.693916,25.201474,0.469392
2,Afyonkarahisar,159,2599,751808.0,325817,118124,Aegean,0,21.149017,34.569997,2.114902
3,Aksaray,55,1942,441136.0,183803,88275,Central Anatolia,0,12.467810,44.022705,1.246781
4,Amasya,46,1684,342242.0,164371,76550,Black Sea,0,13.440782,49.204949,1.344078


---
## Step 8 · Save master.csv

In [22]:
df_merged.to_csv(BASE / 'master.csv', index=False)
print(f'Saved master.csv  →  {df_merged.shape[0]} rows × {df_merged.shape[1]} columns')
print('\nColumns:', df_merged.columns.tolist())

# Missing value summary
missing = df_merged.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print('\nNo missing values in master.csv')
else:
    print('\nMissing values:')
    print(missing)

# --- Major city spot-check ---
# Confirms that merges worked and EV_Count / Total_Stations are plausible
print('\n--- Major city spot-check ---')
SPOT_KEYS = ['istanbul', 'ankara', 'izmir', 'bursa', 'antalya']
spot_mask = df_merged['Province'].apply(key_name).isin(SPOT_KEYS)
cols = ['Province', 'EV_Count', 'Total_Stations', 'Population', 'Region', 'Is_Metropolitan']
print(df_merged.loc[spot_mask, cols].to_string(index=False))

# Quick distribution check — extreme outliers usually signal a data error
print('\n--- Distribution sanity check ---')
print(df_merged[['EV_Count', 'Total_Stations', 'Population', 'Total_Vehicles']]
      .describe()
      .apply(lambda col: col.map(lambda x: f'{x:,.0f}')))

Saved master.csv  →  81 rows × 11 columns

Columns: ['Province', 'Total_Stations', 'EV_Count', 'Population', 'Total_Vehicles', 'Car', 'Region', 'Is_Metropolitan', 'Stations_Per_Capita', 'EVs_Per_10k', 'Stations_Per_10k']

No missing values in master.csv

--- Major city spot-check ---
Province  EV_Count  Total_Stations  Population           Region  Is_Metropolitan
  Ankara     47981            1777   5910320.0 Central Anatolia                1
 Antalya     16715             886   2777677.0    Mediterranean                1
   Bursa     15937             672   3263011.0          Marmara                1
İstanbul     89682            3874  15754053.0          Marmara                1
   İzmir     24012             672   4504185.0           Aegean                1

--- Distribution sanity check ---
      EV_Count Total_Stations  Population Total_Vehicles
count       81             81          81             81
mean     4,742            183   1,062,866        416,687
std     11,489         